# `bea(series, start, end=None)` and `bea_series()`

BEA national accounts series from `indicators.bea`: GDP and its parts, incomes and price indexes, oldest first. `bea_series()` is the map from each series id to its dataset (the NIPA table it comes from), name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its BEA id or by its name, in any case and spacing. Periods with no value are left out, never filled.

In [1]:
from datetime import date, timedelta
from decimal import Decimal

import polars as pl
from IPython.display import Markdown

from arora_data.bea import BeaSeries, BeaValue
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import bea, bea_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest periods with a value.

In [2]:
show(bea_series())

21 rows

series_id|dataset|name|units|frequency|category|first_date|latest_date
---|---|---|---|---|---|---|---
A006RC|bea/nipa_t10105|Gross private domestic investment|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
A006RX|bea/nipa_t10106|Real gross private domestic investment|USD millions (chained)|quarterly|National Accounts|1947-01-01|2026-07-01
A019RC|bea/nipa_t10105|Net exports of goods and services|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
A019RX|bea/nipa_t10106|Real net exports of goods and services|USD millions (chained)|quarterly|National Accounts|1970-01-01|2026-07-01
A032RC|bea/nipa_t11200|National income|USD millions|quarterly|National Accounts — Income|1947-01-01|2026-04-01
A033RC|bea/nipa_t11200|Compensation of employees|USD millions|quarterly|National Accounts — Income|1947-01-01|2026-07-01
A034RC|bea/nipa_t11200|Wages and salaries|USD millions|quarterly|National Accounts — Income|1947-01-01|2026-07-01
A051RC|bea/nipa_t11200|Corporate profits (with IVA & CCAdj)|USD millions|quarterly|National Accounts — Income|1947-01-01|2026-04-01
A191RC|bea/nipa_t10105|Gross domestic product|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
A191RD|bea/nipa_t10109|GDP implicit price deflator|index|quarterly|National Accounts — Prices|1947-01-01|2026-07-01
A191RL|bea/nipa_t10101|Real GDP, percent change (annual rate)|percent change, annual rate|quarterly|National Accounts|1947-04-01|2026-07-01
A191RX|bea/nipa_t10106|Real gross domestic product|USD millions (chained)|quarterly|National Accounts|1947-01-01|2026-07-01
A822RC|bea/nipa_t10105|Government consumption expenditures & gross investment|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
A822RX|bea/nipa_t10106|Real government consumption & gross investment|USD millions (chained)|quarterly|National Accounts|1947-01-01|2026-07-01
B020RC|bea/nipa_t10105|Exports of goods and services|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
B021RC|bea/nipa_t10105|Imports of goods and services|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
DPCCRG|bea/nipa_t20804|Core PCE price index (ex food & energy)|index|monthly|National Accounts — Prices|1959-01-01|2026-09-01
DPCERC|bea/nipa_t10105|Personal consumption expenditures|USD millions|quarterly|National Accounts|1947-01-01|2026-07-01
DPCERD|bea/nipa_t10109|PCE implicit price deflator|index|quarterly|National Accounts — Prices|1947-01-01|2026-07-01
DPCERG|bea/nipa_t20804|PCE price index|index|monthly|National Accounts — Prices|1959-01-01|2026-09-01
DPCERX|bea/nipa_t10106|Real personal consumption expenditures|USD millions (chained)|quarterly|National Accounts|1947-01-01|2026-07-01

## The map between names and ids
Two plain dicts from the same call. Every name is unique, so either one looks a series up.

In [3]:
by_name = {s.name: s.series_id for s in bea_series()}
by_id = {s.series_id: s.name for s in bea_series()}
assert len(by_name) == len(by_id)
print(by_name["Real gross domestic product"], "|", by_id["A051RC"])

A191RX | Corporate profits (with IVA & CCAdj)


## One series, by id or by name
Nominal GDP over the last two years. The id and the name, in any case and spacing, give the same rows.

In [4]:
two_years = today_et() - timedelta(days=730)
assert bea("A191RC", two_years) == bea("a191rc", two_years) == bea("  gross DOMESTIC product ", two_years)
show(bea("A191RC", two_years))

7 rows

series_id|name|date|value|units
---|---|---|---|---
A191RC|Gross domestic product|2025-01-01|30171979.00000000|USD millions
A191RC|Gross domestic product|2025-04-01|30630174.00000000|USD millions
A191RC|Gross domestic product|2025-07-01|31180493.00000000|USD millions
A191RC|Gross domestic product|2025-10-01|31462523.00000000|USD millions
A191RC|Gross domestic product|2026-01-01|31906274.00000000|USD millions
A191RC|Gross domestic product|2026-04-01|32563030.00000000|USD millions
A191RC|Gross domestic product|2026-07-01|33010036.00000000|USD millions

## Units
Dollar levels are USD millions (FMP served GDP in billions), real ones chained USD millions. A191RL, real GDP growth, is a percent change at an annual rate (2.5 is 2.5%). Price indexes are index levels.

In [5]:
last = {s: bea(s, two_years)[-1] for s in ("A191RC", "A191RX", "A191RL", "A191RD")}
for s, v in last.items():
    print(f"{s}  {v.date}  {v.value.normalize():>12}  {v.units}")
print("nominal GDP, USD trillions:", (last["A191RC"].value / Decimal(10**6)).quantize(Decimal("0.01")))

A191RC  2026-07-01      33010036  USD millions
A191RX  2026-07-01      24611077  USD millions (chained)
A191RL  2026-07-01           3.4  percent change, annual rate
A191RD  2026-07-01       134.127  index
nominal GDP, USD trillions: 33.01


## How dates are set
Quarterly series are dated at the quarter's first day (2026Q3 is 2026-07-01). The two PCE price indexes are monthly, dated at the month's first day. National income and corporate profits are first published with BEA's second GDP estimate, about a month after GDP, so for that month they end one quarter earlier.

In [6]:
print([v.date for v in bea("A191RC", date(2025, 1, 1))])
print([v.date for v in bea("DPCCRG", date(2026, 5, 1))])
print({s.series_id: str(s.latest_date) for s in bea_series() if s.series_id in ("A191RC", "A032RC", "A051RC", "DPCCRG")})

[datetime.date(2025, 1, 1), datetime.date(2025, 4, 1), datetime.date(2025, 7, 1), datetime.date(2025, 10, 1), datetime.date(2026, 1, 1), datetime.date(2026, 4, 1), datetime.date(2026, 7, 1)]


[datetime.date(2026, 5, 1), datetime.date(2026, 6, 1), datetime.date(2026, 7, 1), datetime.date(2026, 8, 1), datetime.date(2026, 9, 1)]


{'A032RC': '2026-04-01', 'A051RC': '2026-04-01', 'A191RC': '2026-07-01', 'DPCCRG': '2026-09-01'}


## Full history in one call
From 1900 there is no row limit; a series starts at its own first value. Real GDP growth from 1947, with the quarters it shrank, by decade. BEA's growth rate needs a prior quarter, so it starts one quarter after the levels.

In [7]:
growth = to_df(bea("A191RL", date(1900, 1, 1)), BeaValue)
print(f"{growth.height:,} quarters, {growth['date'].min()} to {growth['date'].max()}")
growth.group_by(pl.col("date").dt.year() // 10 * 10).agg((pl.col("value") < 0).sum().alias("quarters shrinking"), pl.len().alias("quarters")).sort("date").rename({"date": "decade"})

318 quarters, 1947-04-01 to 2026-07-01


decade,quarters shrinking,quarters
i32,u32,u32
1940,5,11
1950,8,40
1960,3,40
1970,7,40
1980,6,40
1990,2,40
2000,7,40
2010,3,40
2020,3,27


## Revisions
Only BEA's latest estimate is kept. The main repo re-reads every table's whole history each weekday, so when BEA revises a quarter the old value is replaced, and earlier estimates are not stored.

## Not here
Monthly nominal and real PCE are `fred("PCE")` and `fred("PCEC96")`. The unemployment rate and payrolls are `bls()`.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [8]:
for args in [("GDP", date(2026, 1, 1)), ("gross domestic", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("A191RC", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        bea(*args)
    except ValueError as e:
        print(e)

'GDP' is not a stored BEA series: use an id or name from bea_series(), such as A191RC or 'Gross domestic product'


'gross domestic' is not a stored BEA series: use an id or name from bea_series(), such as A191RC or 'Gross domestic product'


None is not a stored BEA series: use an id or name from bea_series(), such as A191RC or 'Gross domestic product'
start must be on or before end
